# Notebook B · Model AI pendeteksi URL phishing
**Matrikulasi MTI UNLA · Sesi 2 · Versi mahasiswa**

Di kelas, Anda menebak sendiri email mana yang phishing. Sekarang giliran mesin. Kita akan:
1. melihat bagaimana mesin "membaca" sebuah URL (fitur),
2. melatih dua model,
3. menilai model dengan **confusion matrix**, **precision**, dan **recall**,
4. melihat jebakan akurasi saat phishing jarang,
5. mencoba mengecoh model (*evasion*).

Jalankan sel dari atas ke bawah dengan **Shift + Enter**. Ganti setiap `___` dengan jawaban Anda. Kalau muncul error yang menyebut `___`, berarti ada bagian yang belum diisi.

Data di notebook ini **sintetis**. Nama domain dan bank adalah contoh untuk latihan; alamat IP memakai blok dokumentasi.

## Langkah 0 · Siapkan data
Sel ini membuat 2.000 URL berlabel (`1` = phishing, `0` = aman). Cukup jalankan.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

domain_aman = ["unla.ac.id", "siakad.unla.ac.id", "jurnal.unla.ac.id", "tokobuku-contoh.co.id",
               "beritaharian-contoh.id", "pemkot-contoh.go.id", "banknusantara.co.id", "github.com",
               "perpustakaan-contoh.id", "kampus-contoh.ac.id"]
jalur_aman = ["", "/", "/berita/2026/10/jadwal-kuliah", "/produk/buku-python", "/layanan/pajak",
              "/repo/tugas-akhir", "/artikel/view/123", "/profil", "/login", "/akun/pengaturan",
              "/pengumuman/matrikulasi", "/search?q=keamanan+siber"]

domain_palsu = ["unIa.ac.id", "unla-ac.id", "banknusantara-verifikasi.top", "bank-nusantara.secure-login.xyz",
                "promo-dompetku.top", "paketkilat-id.com", "siakad-unla.online", "akun-github.support"]
jalur_palsu = ["/verifikasi", "/login", "/akun/blokir", "/hadiah/klaim", "/update-data", "/bayar-bea",
               "/konfirmasi-password", "/secure/login.php", "/reset"]

def buat_url_aman():
    skema = "https://" if rng.random() < 0.9 else "http://"
    return skema + rng.choice(domain_aman) + rng.choice(jalur_aman)

def buat_url_palsu():
    skema = "https://" if rng.random() < 0.35 else "http://"
    r = rng.random()
    if r < 0.25:
        host = f"198.51.100.{int(rng.integers(2, 250))}"
    elif r < 0.35:
        host = "banknusantara.co.id@" + rng.choice(domain_palsu)
    else:
        host = rng.choice(domain_palsu)
    jalur = rng.choice(jalur_palsu) if rng.random() < 0.8 else rng.choice(["/", "/berita", "/profil"])
    return skema + host + jalur

n_palsu, n_aman = 600, 1400
data = pd.DataFrame({
    "url": [buat_url_palsu() for _ in range(n_palsu)] + [buat_url_aman() for _ in range(n_aman)],
    "phishing": [1] * n_palsu + [0] * n_aman,
}).sample(frac=1, random_state=1).reset_index(drop=True)
data.head(8)

,url,phishing
0,https://jurnal.unla.ac.id/berita/2026/10/jadwa...,0
1,https://beritaharian-contoh.id/artikel/view/123,0
2,https://tokobuku-contoh.co.id/profil,0
3,https://pemkot-contoh.go.id/berita/2026/10/jad...,0
4,http://unla.ac.id/artikel/view/123,0
5,http://unla.ac.id,0
6,https://unla.ac.id/produk/buku-python,0
7,https://unla-ac.id/login,1


## Langkah 1 · Bagaimana mesin melihat URL: fitur
Mesin tidak "membaca" seperti manusia. Kita ubah setiap URL menjadi beberapa angka (**fitur**), misalnya panjangnya, apakah memakai alamat IP, dan apakah ada kata yang sering dipakai penipu.

In [5]:
import re

KATA_MENCURIGAKAN = ["login", "verifikasi", "blokir", "hadiah", "update", "bayar", "password", "reset", "secure"]

def ekstrak_fitur(url):
    host = re.sub(r"^https?://", "", url).split("/")[0]
    return {
        "panjang": len(url),
        "jumlah_titik": url.count("."),
        "jumlah_tanda_hubung": host.count("-"),
        "pakai_ip": int(bool(re.fullmatch(r"\d+\.\d+\.\d+\.\d+", host.split("@")[-1]))),
        "ada_at": int("@" in url),
        "pakai_https": int(url.startswith("https://")),
        "kata_mencurigakan": sum(k in url.lower() for k in KATA_MENCURIGAKAN),
        "tld_aneh": int(host.endswith((".top", ".xyz", ".online", ".support"))),
    }

contoh = "http://198.51.100.7/bank-nusantara/verifikasi-login"
print(contoh)
pd.Series(ekstrak_fitur(contoh))

http://198.51.100.7/bank-nusantara/verifikasi-login


,0
panjang,51
jumlah_titik,3
jumlah_tanda_hubung,0
pakai_ip,1
ada_at,0
pakai_https,0
kata_mencurigakan,2
tld_aneh,0


In [19]:
X = pd.DataFrame([ekstrak_fitur(u) for u in data["url"]])
y = data["phishing"]
X.head()

,panjang,jumlah_titik,jumlah_tanda_hubung,pakai_ip,ada_at,pakai_https,kata_mencurigakan,tld_aneh
0,54,3,0,0,0,1,0,0
1,47,1,1,0,0,1,0,0
2,36,2,1,0,0,1,0,0
3,56,2,1,0,0,1,0,0
4,34,2,0,0,0,0,0,0


## Langkah 2 · Pisahkan latihan soal dan ujian
Ingat analogi di kelas: **data latih = latihan soal**, **data uji = ujian**. Model tidak boleh melihat data uji saat belajar.

Isi `0.3` dengan proporsi data uji: 30% ditulis sebagai `0.3`.

In [23]:
from sklearn.model_selection import train_test_split

X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

print("Data latih:", len(X_latih), "URL | Data uji:", len(X_uji), "URL")

Data latih: 1400 URL | Data uji: 600 URL


## Langkah 3 · Latih dua model
- **Regresi logistik**: memberi bobot pada setiap fitur.
- **Pohon keputusan**: membuat serangkaian pertanyaan ya/tidak, mirip cara manusia memutuskan.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text

model_logistik = LogisticRegression(max_iter=1000).fit(X_latih, y_latih)
model_pohon = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_latih, y_latih)

print("Aturan yang dipelajari pohon keputusan:\n")
print(export_text(model_pohon, feature_names=list(X.columns)))

Aturan yang dipelajari pohon keputusan:

|--- kata_mencurigakan <= 0.50
|   |--- tld_aneh <= 0.50
|   |   |--- pakai_ip <= 0.50
|   |   |   |--- class: 0
|   |   |--- pakai_ip >  0.50
|   |   |   |--- class: 1
|   |--- tld_aneh >  0.50
|   |   |--- class: 1
|--- kata_mencurigakan >  0.50
|   |--- panjang <= 36.50
|   |   |--- pakai_https <= 0.50
|   |   |   |--- class: 1
|   |   |--- pakai_https >  0.50
|   |   |   |--- class: 0
|   |--- panjang >  36.50
|   |   |--- class: 1



## Langkah 4 · Nilai model di data uji
Isi "model_pohon" dengan nama variabel model pohon keputusan dari Langkah 3.

In [32]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score

def laporan(nama, model, X_eval, y_eval):
    pred = model.predict(X_eval)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred, labels=[0, 1]).ravel()

    print(f"\n== {nama} ==")
    print(f"  Tertangkap (TP): {tp:5d}   Lolos (FN): {fn:5d}")
    print(f"  Alarm palsu (FP): {fp:5d}   Benar aman (TN): {tn:5d}")
    print(f"  Akurasi  : {accuracy_score(y_eval, pred):.1%}")
    print(f"  Precision: {precision_score(y_eval, pred, zero_division=0):.1%}")
    print(f"  Recall   : {recall_score(y_eval, pred, zero_division=0):.1%}")
    print(f"  F1       : {f1_score(y_eval, pred, zero_division=0):.1%}")

laporan("Regresi logistik", model_logistik, X_uji, y_uji)
laporan("Pohon keputusan", model_pohon, X_uji, y_uji)


== Regresi logistik ==
  Tertangkap (TP):   164   Lolos (FN):    16
  Alarm palsu (FP):     2   Benar aman (TN):   418
  Akurasi  : 97.0%
  Precision: 98.8%
  Recall   : 91.1%
  F1       : 94.8%

== Pohon keputusan ==
  Tertangkap (TP):   147   Lolos (FN):    33
  Alarm palsu (FP):     2   Benar aman (TN):   418
  Akurasi  : 94.2%
  Precision: 98.7%
  Recall   : 81.7%
  F1       : 89.4%


### Pertanyaan
1. Model mana yang lebih sedikit meloloskan phishing (FN)?
   Model Regresi Logistik lebih sedikit meloloskan phishing dibandingkan Pohon Keputusan. Regresi Logistik memiliki False Negative (FN) sebanyak 16, sedangkan Pohon Keputusan sebanyak 33. Artinya, Regresi Logistik lebih baik dalam mendeteksi URL phishing agar tidak salah dianggap aman.

2. Bandingkan dengan confusion matrix kelas Anda di Sesi 2. Siapa yang lebih teliti, kelas atau mesin? Kenapa menurut Anda?
   Berdasarkan hasil pengujian, model Regresi Logistik memiliki akurasi 97%, precision 98,8%, dan recall 91,1%. Model ini mampu mendeteksi 164 dari 180 URL phishing.

   Menurut saya, mesin lebih cepat dan konsisten dalam menganalisis URL, tetapi hasilnya perlu dibandingkan dengan confusion matrix kelas. Pemeriksaan manusia tetap diperlukan karena mesin masih bisa melakukan kesalahan.

## Langkah 5 · Jebakan akurasi saat phishing jarang
Di dunia nyata phishing **jarang**. Kita uji model pada 10.000 URL baru yang hanya **1%** di antaranya phishing.

Isi 100 dengan jumlah URL phishing: 1% dari 10.000.

In [34]:
rng = np.random.default_rng(99)
n_phishing_nyata = 100
dunia_nyata = pd.DataFrame({
    "url": [buat_url_palsu() for _ in range(n_phishing_nyata)] + [buat_url_aman() for _ in range(10_000 - n_phishing_nyata)],
    "phishing": [1] * n_phishing_nyata + [0] * (10_000 - n_phishing_nyata),
})
X_nyata = pd.DataFrame([ekstrak_fitur(u) for u in dunia_nyata["url"]])
laporan("Regresi logistik di dunia nyata (1% phishing)", model_logistik, X_nyata, dunia_nyata["phishing"])

class SelaluAman:
    def predict(self, X):
        return np.zeros(len(X), dtype=int)

laporan("Model malas: selalu menjawab 'aman'", SelaluAman(), X_nyata, dunia_nyata["phishing"])


== Regresi logistik di dunia nyata (1% phishing) ==
  Tertangkap (TP):    87   Lolos (FN):    13
  Alarm palsu (FP):    79   Benar aman (TN):  9821
  Akurasi  : 99.1%
  Precision: 52.4%
  Recall   : 87.0%
  F1       : 65.4%

== Model malas: selalu menjawab 'aman' ==
  Tertangkap (TP):     0   Lolos (FN):   100
  Alarm palsu (FP):     0   Benar aman (TN):  9900
  Akurasi  : 99.0%
  Precision: 0.0%
  Recall   : 0.0%
  F1       : 0.0%


**Perhatikan:** bandingkan precision regresi logistik di Langkah 4 dengan di Langkah 5. Modelnya sama, tetapi karena phishing kini jarang, porsi alarm palsu di antara peringatannya jauh lebih besar. Lalu lihat model malas yang tidak pernah menangkap phishing satu pun tetap mendapat akurasi 99%. Karena itu analis keamanan melihat **precision dan recall**, bukan akurasi saja. (Jawaban soal 3 kuis pembuka: *belum tentu*.)

## Langkah 6 · Coba kecoh modelnya (*evasion*)
Penyerang yang tahu fitur apa yang dilihat model bisa mengubah URL-nya. Ubah URL di bawah supaya model menganggapnya **aman**, tanpa mengubah tujuan penipuannya. Petunjuk: lihat aturan pohon keputusan di Langkah 3.

In [36]:
asli = "http://198.51.100.7/bank-nusantara/verifikasi-login"
ubahan = "https://banknusantara.co.id/login"

for url in [asli, ubahan]:
    f = pd.DataFrame([ekstrak_fitur(url)])
    p = model_logistik.predict_proba(f)[0, 1]
    print(f"{url}\n  peluang phishing menurut model: {p:.0%}\n")

http://198.51.100.7/bank-nusantara/verifikasi-login
  peluang phishing menurut model: 100%

https://banknusantara.co.id/login
  peluang phishing menurut model: 17%



---
**Selesai.** Anda sudah membangun dan menilai model AI keamanan pertama Anda, lalu mengecohnya. Dua arah yang sama akan Anda dalami di MTI: **AI untuk keamanan siber** (MK03, MK05) dan **keamanan sistem AI** (MK13, MK16).